<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/05_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import gc
import random
import pandas as pd
import numpy as np

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

S3_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s3_cleaned.tsv"
)

GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

FEATURE_OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Fuzzy_Features"
)

os.makedirs(FEATURE_OUTPUT_ROOT, exist_ok=True)

print("Setup complete.")

Setup complete.


In [9]:
gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype="string"
)

gt_lookup = {}

for row in gt.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched) or matched == "":
        gt_lookup[s1_id] = set()
    else:
        gt_lookup[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }

print("Ground-truth S1 entities:", len(gt_lookup))

del gt
gc.collect()

Ground-truth S1 entities: 2206821


0

In [10]:
s1_features = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_norm",
        "name_core",
        "address_norm"
    ],
    dtype="string"
)

print("S1 shape:", s1_features.shape)

S1 shape: (2206821, 5)


In [4]:
import os
import gc
import pandas as pd
import numpy as np

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

s1_features = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_norm",
        "name_core",
        "address_norm"
    ],
    dtype="string"
)

print("S1 shape:", s1_features.shape)

S1 shape: (2206821, 5)


In [11]:
import random

positive_pairs = []

rng = random.Random(42)

for row in s1_features.itertuples(index=False):

    s1_id = row.entity_id

    true_s2 = [
        x for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-")
    ]

    if not true_s2:
        continue
    if len(true_s2) > 3:
        true_s2 = rng.sample(true_s2, 3)

    for s2_id in true_s2:
        positive_pairs.append({
            "s1_entity_id": s1_id,
            "s2_entity_id": s2_id,
            "label": 1
        })

positive_pairs = pd.DataFrame(positive_pairs)

print("Positive pairs:", len(positive_pairs))
print(positive_pairs.head())

Positive pairs: 3526233
   s1_entity_id  s2_entity_id  label
0  S1-925783039  S2-517291332      1
1  S1-925783039  S2-157377754      1
2  S1-773889195  S2-970528089      1
3  S1-773889195  S2-374107005      1
4  S1-377745466  S2-203037501      1


In [13]:
# ---------------------------------------------------------
# Create same-country negative pairs
# ---------------------------------------------------------

rng = np.random.default_rng(42)

# Sample S1 entities
sample_size = min(20000, len(s1_features))

sample_indices = rng.choice(
    len(s1_features),
    size=sample_size,
    replace=False
)

sample_s1 = s1_features.iloc[sample_indices].copy()

# Group sampled S1 IDs by country
s1_by_country = {}

for row in sample_s1[
    ["entity_id", "country_norm"]
].itertuples(index=False):

    if pd.isna(row.country_norm):
        continue

    s1_by_country.setdefault(
        row.country_norm,
        []
    ).append(row.entity_id)


# We will collect a pool of S2 IDs for each country.
# Only IDs are stored, so memory usage stays reasonable.

s2_ids_by_country = {}

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "country_norm"],
    chunksize=200_000,
    dtype="string"
):

    for row in chunk.itertuples(index=False):

        if pd.isna(row.country_norm):
            continue

        if row.country_norm in s1_by_country:

            s2_ids_by_country.setdefault(
                row.country_norm,
                []
            ).append(row.entity_id)

    del chunk
    gc.collect()


print("S2 country pools:")

for country, ids in s2_ids_by_country.items():
    print(country, len(ids))


# ---------------------------------------------------------
# Generate negative pairs
# ---------------------------------------------------------

negative_pairs = []

for row in sample_s1[
    ["entity_id", "country_norm"]
].itertuples(index=False):

    s1_id = row.entity_id
    country = row.country_norm

    if pd.isna(country):
        continue

    pool = s2_ids_by_country.get(country, [])

    if not pool:
        continue

    true_s2 = gt_lookup.get(s1_id, set())

    # Randomly choose more than needed because
    # some selected IDs may be true matches.
    candidate_indices = rng.choice(
        len(pool),
        size=min(10, len(pool)),
        replace=False
    )

    added = 0

    for idx in candidate_indices:

        s2_id = pool[idx]

        # Never label a known true match as negative
        if s2_id in true_s2:
            continue

        negative_pairs.append({
            "s1_entity_id": s1_id,
            "s2_entity_id": s2_id,
            "label": 0
        })

        added += 1

        if added >= 2:
            break


negative_pairs = pd.DataFrame(
    negative_pairs
)

print("\nNegative pairs:", len(negative_pairs))
print(negative_pairs.head())

S2 country pools:
india 2017799
us 3016817

Negative pairs: 40000
   s1_entity_id  s2_entity_id  label
0  S1-120325152  S2-627599221      0
1  S1-120325152  S2-210441197      0
2  S1-832156594  S2-645062595      0
3  S1-832156594  S2-840933135      0
4  S1-812081039  S2-341865299      0


In [12]:
s2_lookup = {}

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_norm",
        "name_core",
        "address_norm"
    ],
    chunksize=200_000,
    dtype="string"
):

    for row in chunk.itertuples(index=False):
        s2_lookup[row.entity_id] = {
            "country_norm": row.country_norm,
            "name_norm": row.name_norm,
            "name_core": row.name_core,
            "address_norm": row.address_norm
        }

    del chunk
    gc.collect()

print("S2 records loaded:", len(s2_lookup))

S2 records loaded: 5034616


In [14]:
positive_sample = positive_pairs.sample(
    n=min(40000, len(positive_pairs)),
    random_state=42
).reset_index(drop=True)

negative_sample = negative_pairs.copy()

print("Positive sample:", len(positive_sample))
print("Negative sample:", len(negative_sample))

Positive sample: 40000
Negative sample: 40000


In [15]:
feature_pairs = pd.concat(
    [
        positive_sample,
        negative_sample
    ],
    ignore_index=True
)

feature_pairs = feature_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Total feature pairs:", len(feature_pairs))
print(feature_pairs["label"].value_counts())

Total feature pairs: 80000
label
0    40000
1    40000
Name: count, dtype: int64


In [16]:
s2_features = pd.DataFrame.from_dict(
    s2_lookup,
    orient="index"
).reset_index()

s2_features = s2_features.rename(
    columns={"index": "entity_id"}
)

print("S2 feature rows:", len(s2_features))
print(s2_features.head())

S2 feature rows: 5034616
      entity_id country_norm                       name_norm  \
0  S2-166376419        india  र म म र क ट ग प र इव ट ल म ट ड   
1  S2-764573417           us       holloway peak inc seafood   
2  S2-639257739        india         आद त य प र पर ट ज एलएलप   
3  S2-163963287           us                      summit inc   
4   S2-49942811           us    delta tetlecommunication inc   

                        name_core  \
0  र म म र क ट ग प र इव ट ल म ट ड   
1       holloway peak inc seafood   
2         आद त य प र पर ट ज एलएलप   
3                          summit   
4        delta tetlecommunication   

                                    address_norm  
0        kh no 570 13 new delhi west delhi delhi  
1                        105 elm st morganton nc  
2  g 3 571 gulmohar colony bhopal madhya pradesh  
3            greensboro nc 19 1 2 stardust trail  
4                  914 pierpont ave cleveland oh  


In [18]:
s1_pair_data = s1_features.rename(
    columns={
        "entity_id": "s1_entity_id",
        "country_norm": "s1_country",
        "name_norm": "s1_name_norm",
        "name_core": "s1_name_core",
        "address_norm": "s1_address_norm"
    }
)

s2_pair_data = s2_features.rename(
    columns={
        "entity_id": "s2_entity_id",
        "country_norm": "s2_country",
        "name_norm": "s2_name_norm",
        "name_core": "s2_name_core",
        "address_norm": "s2_address_norm"
    }
)

In [19]:
feature_dataset = feature_pairs.merge(
    s1_pair_data,
    on="s1_entity_id",
    how="left"
)

feature_dataset = feature_dataset.merge(
    s2_pair_data,
    on="s2_entity_id",
    how="left"
)

print("Feature dataset shape:", feature_dataset.shape)

Feature dataset shape: (80000, 11)


In [20]:
!pip install -q rapidfuzz
from rapidfuzz import fuzz
import unicodedata
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.7 MB/s eta 0:00:00


In [21]:
def clean_text(v):
    if v is None or pd.isna(v):
        return ""
    return str(v).strip()


def char_similarity(a, b):
    a = clean_text(a)
    b = clean_text(b)

    if not a or not b:
        return np.nan

    return fuzz.ratio(a, b) / 100.0


def token_jaccard(a, b):
    A = set(clean_text(a).split())
    B = set(clean_text(b).split())

    if not A or not B:
        return np.nan

    return len(A & B) / len(A | B)


def ngram_jaccard(a, b, n=3):
    a = clean_text(a).replace(" ", "")
    b = clean_text(b).replace(" ", "")

    if not a or not b:
        return np.nan

    A = {
        a[i:i+n]
        for i in range(max(1, len(a) - n + 1))
    }

    B = {
        b[i:i+n]
        for i in range(max(1, len(b) - n + 1))
    }

    if not A or not B:
        return np.nan

    return len(A & B) / len(A | B)

In [23]:
!pip install -q indic-transliteration

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 4.7 MB/s eta 0:00:00


In [24]:
import unicodedata

from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate


SCRIPT_MAP = {
    "DEVANAGARI": sanscript.DEVANAGARI,
    "TELUGU": sanscript.TELUGU,
    "KANNADA": sanscript.KANNADA,
    "TAMIL": sanscript.TAMIL,
    "GUJARATI": sanscript.GUJARATI,
    "BENGALI": sanscript.BENGALI,
    "MALAYALAM": sanscript.MALAYALAM,
    "ORIYA": sanscript.ORIYA,
    "GURMUKHI": sanscript.GURMUKHI,
}


def normalize_transliteration(text):

    if pd.isna(text):
        return ""

    text = str(text)

    scripts_found = set()

    for ch in text:

        try:
            char_name = unicodedata.name(ch)
        except ValueError:
            continue

        for script_name in SCRIPT_MAP:

            if script_name in char_name:
                scripts_found.add(script_name)

    # Already Latin
    if not scripts_found:
        return text.lower()

    result = text

    for script_name in scripts_found:

        source_script = SCRIPT_MAP[script_name]

        try:
            result = transliterate(
                result,
                source_script,
                sanscript.IAST
            )
        except Exception:
            pass

    # Remove diacritics
    result = "".join(
        ch
        for ch in unicodedata.normalize("NFKD", result)
        if not unicodedata.combining(ch)
    )

    result = result.lower()

    # Normalize whitespace
    result = " ".join(result.split())

    return result

In [25]:
test_names = [
    "ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड",
    "ಎಸ್‌ಎಸ್ ಸಿಸ್ಟಮ್ಸ್ ಲಿಮಿಟೆಡ್",
    "શ્રી ઇન્ડસ્ટ્રીઝ પ્રાઇવેટ લિમિટેડ",
    "குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்",
    "green logistics"
]

for name in test_names:
    print(name)
    print(" -> ", normalize_transliteration(name))
    print()

ग्रीन लॉजिस्टिक्स प्राइवेट लिमिटेड
 ->  grina laॉjistiksa praiveta limiteda

ಎಸ್‌ಎಸ್ ಸಿಸ್ಟಮ್ಸ್ ಲಿಮಿಟೆಡ್
 ->  es‌es sistams limited

શ્રી ઇન્ડસ્ટ્રીઝ પ્રાઇવેટ લિમિટેડ
 ->  sri indastrijha praiveta limiteda

குளோபல் பிசினஸ் பிரைவேட் லிமிடெட்
 ->  ghulobhal bhijhinas bhiraivedh limidhedh

green logistics
 ->  green logistics



In [26]:
feature_dataset["s2_name_translit"] = (
    feature_dataset["s2_name_norm"]
    .fillna("")
    .apply(normalize_transliteration)
)

In [28]:
# Create transliterated S1 and S2 names

feature_dataset["s1_name_translit"] = (
    feature_dataset["s1_name_norm"]
    .fillna("")
    .astype(str)
    .str.lower()
)

feature_dataset["s2_name_translit"] = (
    feature_dataset["s2_name_norm"]
    .fillna("")
    .apply(normalize_transliteration)
)

print(
    feature_dataset[
        [
            "s1_name_norm",
            "s2_name_norm",
            "s1_name_translit",
            "s2_name_translit"
        ]
    ].head(10).to_string(index=False)
)

                   s1_name_norm                      s2_name_norm                s1_name_translit                      s2_name_translit
                smith sher leon                         bison llc                 smith sher leon                             bison llc
                jain energy llp    vansh facility exports exports                 jain energy llp        vansh facility exports exports
    rosh agencies india pvt ltd          పర ఫ క ట బ జ న స ఎల ఎల ప     rosh agencies india pvt ltd para pha ka ta ba ja na sa ela ela pa
     new delhi ventures pvt ltd                    zjt online pvt      new delhi ventures pvt ltd                        zjt online pvt
                  yz taiwan llc                  highland allegro                   yz taiwan llc                      highland allegro
 shivam maa engineering pvt ltd    shivam maa engineering pvt ltd  shivam maa engineering pvt ltd        shivam maa engineering pvt ltd
              torres dragon llc valentia reicher

In [29]:
feature_dataset["name_char_similarity"] = feature_dataset.apply(
    lambda r: char_similarity(
        r["s1_name_norm"],
        r["s2_name_norm"]
    ),
    axis=1
)

feature_dataset["name_3gram_jaccard"] = feature_dataset.apply(
    lambda r: ngram_jaccard(
        r["s1_name_norm"],
        r["s2_name_norm"],
        3
    ),
    axis=1
)

feature_dataset["name_token_jaccard"] = feature_dataset.apply(
    lambda r: token_jaccard(
        r["s1_name_norm"],
        r["s2_name_norm"]
    ),
    axis=1
)

feature_dataset["address_char_similarity"] = feature_dataset.apply(
    lambda r: char_similarity(
        r["s1_address_norm"],
        r["s2_address_norm"]
    ),
    axis=1
)

feature_dataset["address_token_jaccard"] = feature_dataset.apply(
    lambda r: token_jaccard(
        r["s1_address_norm"],
        r["s2_address_norm"]
    ),
    axis=1
)

feature_dataset["country_exact"] = (
    feature_dataset["s1_country"].fillna("").str.lower()
    ==
    feature_dataset["s2_country"].fillna("").str.lower()
).astype(int)

feature_dataset["translit_name_char_similarity"] = feature_dataset.apply(
    lambda r: char_similarity(
        r["s1_name_translit"],
        r["s2_name_translit"]
    ),
    axis=1
)

print("Feature calculation complete.")

Feature calculation complete.


In [ ]:
FEATURE_COLS = [
    "name_char_similarity",
    "name_3gram_jaccard",
    "name_token_jaccard",
    "address_char_similarity",
    "address_token_jaccard",
    "country_exact",
    "translit_name_char_similarity"
]

display(
    feature_dataset[
        ["label"] + FEATURE_COLS
    ].head(20)
)

In [30]:
FEATURE_COLS = [
    "name_char_similarity",
    "name_3gram_jaccard",
    "name_token_jaccard",
    "address_char_similarity",
    "address_token_jaccard",
    "country_exact",
    "translit_name_char_similarity"
]

display(
    feature_dataset[
        ["label"] + FEATURE_COLS
    ].head(20)
)

,label,name_char_similarity,name_3gram_jaccard,name_token_jaccard,address_char_similarity,address_token_jaccard,country_exact,translit_name_char_similarity
0,0,0.333333,0.000000,0.000000,0.419753,0.000000,1,0.333333
1,0,0.311111,0.000000,0.000000,0.391061,0.052632,1,0.311111
2,0,0.156863,0.000000,0.000000,0.403509,0.050000,1,0.312500
3,0,0.450000,0.034483,0.142857,0.384615,0.105263,1,0.450000
4,0,0.413793,0.000000,0.000000,0.333333,0.000000,1,0.413793
5,1,1.000000,1.000000,1.000000,0.967213,0.916667,1,1.000000
6,0,0.400000,0.027778,0.100000,0.320000,0.000000,1,0.400000
7,0,0.260870,0.000000,0.000000,0.357143,0.000000,1,0.260870
8,1,0.842105,0.578947,0.200000,0.755556,0.800000,1,0.842105
9,0,0.307692,0.000000,0.000000,0.250000,0.000000,1,0.307692


In [31]:
summary = (
    feature_dataset[
        ["label"] + FEATURE_COLS
    ]
    .groupby("label")
    .mean(numeric_only=True)
    .T
)

print(summary.to_string())

label                                 0         1
name_char_similarity           0.323416  0.795814
name_3gram_jaccard             0.019226  0.658732
name_token_jaccard             0.041257  0.623358
address_char_similarity        0.351710  0.836225
address_token_jaccard          0.021920  0.776738
country_exact                  1.000000  1.000000
translit_name_char_similarity  0.341719  0.828650


In [32]:
summary_median = (
    feature_dataset[
        ["label"] + FEATURE_COLS
    ]
    .groupby("label")
    .median(numeric_only=True)
    .T
)

print("\nMEDIANS")
print(summary_median.to_string())


MEDIANS
label                                 0         1
name_char_similarity           0.325581  0.888889
name_3gram_jaccard             0.000000  0.727273
name_token_jaccard             0.000000  0.666667
address_char_similarity        0.349206  0.883721
address_token_jaccard          0.000000  0.800000
country_exact                  1.000000  1.000000
translit_name_char_similarity  0.333333  0.888889


In [35]:
NAME_STOPWORDS = {
    "inc", "incorporated", "llc", "ltd", "limited",
    "pvt", "private", "company", "co", "corp",
    "corporation", "group", "services", "partners",
    "holdings", "associates", "center", "india",
    "and", "of"
}

def meaningful_tokens(name):
    if pd.isna(name) or not str(name).strip():
        return []

    return [
        token
        for token in str(name).split()
        if token not in NAME_STOPWORDS
        and len(token) >= 2
    ]


relevant_tokens = set()

for name in sample_s1["name_core"]:
    relevant_tokens.update(
        meaningful_tokens(name)
    )

print("Relevant S1 tokens:", len(relevant_tokens))

Relevant S1 tokens: 11660


In [36]:
from collections import defaultdict

hard_negative_index = defaultdict(list)

for row in s2_features[
    ["entity_id", "country_norm", "name_core"]
].itertuples(index=False):

    if pd.isna(row.name_core):
        continue

    tokens = set(
        meaningful_tokens(row.name_core)
    )

    relevant = tokens.intersection(
        relevant_tokens
    )

    for token in relevant:
        hard_negative_index[token].append(
            row.entity_id
        )

print("Temporary token index built.")
print("Indexed tokens:", len(hard_negative_index))

Temporary token index built.
Indexed tokens: 11646


In [38]:
s2_country_lookup = dict(
    zip(
        s2_features["entity_id"],
        s2_features["country_norm"]
    )
)

print("S2 country lookup:", len(s2_country_lookup))

S2 country lookup: 5034616


In [39]:
hard_negative_pairs = []

rng = random.Random(42)

for row in sample_s1[
    ["entity_id", "country_norm", "name_core"]
].itertuples(index=False):

    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core

    if pd.isna(name):
        continue

    true_s2 = gt_lookup.get(s1_id, set())

    candidate_ids = set()

    for token in set(meaningful_tokens(name)):
        candidate_ids.update(
            hard_negative_index.get(token, [])
        )

    # Same country + not a true match
    candidates = [
        s2_id
        for s2_id in candidate_ids
        if s2_country_lookup.get(s2_id) == country
        and s2_id not in true_s2
    ]

    if not candidates:
        continue

    if len(candidates) > 2:
        candidates = rng.sample(candidates, 2)

    for s2_id in candidates:
        hard_negative_pairs.append({
            "s1_entity_id": s1_id,
            "s2_entity_id": s2_id,
            "label": 0
        })

print("Hard negative pairs:", len(hard_negative_pairs))

Hard negative pairs: 39926


In [40]:
hard_negative_df = pd.DataFrame(hard_negative_pairs)

print(hard_negative_df.shape)
print(hard_negative_df["label"].value_counts())

(39926, 3)
label
0    39926
Name: count, dtype: int64


In [41]:
positive_sample_hard = positive_pairs.sample(
    n=40000,
    random_state=42
).copy()

print(positive_sample_hard.shape)
print(positive_sample_hard["label"].value_counts())

(40000, 3)
label
1    40000
Name: count, dtype: int64


In [42]:
hard_pair_dataset = pd.concat(
    [
        positive_sample_hard,
        hard_negative_df
    ],
    ignore_index=True
)

hard_pair_dataset = hard_pair_dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(hard_pair_dataset.shape)
print(hard_pair_dataset["label"].value_counts())

(79926, 3)
label
1    40000
0    39926
Name: count, dtype: int64


In [43]:
# Prepare S1 lookup data
s1_pair_data = s1_features.rename(columns={
    "entity_id": "s1_entity_id",
    "country_norm": "s1_country",
    "name_norm": "s1_name_norm",
    "name_core": "s1_name_core",
    "address_norm": "s1_address_norm"
})

# Prepare S2 lookup data
s2_pair_data = s2_features.rename(columns={
    "entity_id": "s2_entity_id",
    "country_norm": "s2_country",
    "name_norm": "s2_name_norm",
    "name_core": "s2_name_core",
    "address_norm": "s2_address_norm"
})

hard_feature_dataset = hard_pair_dataset.merge(
    s1_pair_data,
    on="s1_entity_id",
    how="left"
).merge(
    s2_pair_data,
    on="s2_entity_id",
    how="left"
)

print("Shape:", hard_feature_dataset.shape)
print("Missing S1 rows:", hard_feature_dataset["s1_name_norm"].isna().sum())
print("Missing S2 rows:", hard_feature_dataset["s2_name_norm"].isna().sum())

Shape: (79926, 11)
Missing S1 rows: 0
Missing S2 rows: 0


In [45]:
# Create transliterated names for the hard-pair dataset

hard_feature_dataset["s1_name_translit"] = (
    hard_feature_dataset["s1_name_norm"]
    .fillna("")
    .astype(str)
    .str.lower()
)

hard_feature_dataset["s2_name_translit"] = (
    hard_feature_dataset["s2_name_norm"]
    .fillna("")
    .astype(str)
    .apply(normalize_transliteration)
)

print("Transliteration columns created.")

Transliteration columns created.


In [46]:
# Calculate pairwise similarity features

hard_feature_dataset["name_char_similarity"] = [
    char_similarity(a, b)
    for a, b in zip(
        hard_feature_dataset["s1_name_norm"],
        hard_feature_dataset["s2_name_norm"]
    )
]

hard_feature_dataset["name_3gram_jaccard"] = [
    ngram_jaccard(a, b, 3)
    for a, b in zip(
        hard_feature_dataset["s1_name_norm"],
        hard_feature_dataset["s2_name_norm"]
    )
]

hard_feature_dataset["name_token_jaccard"] = [
    token_jaccard(a, b)
    for a, b in zip(
        hard_feature_dataset["s1_name_norm"],
        hard_feature_dataset["s2_name_norm"]
    )
]

hard_feature_dataset["address_char_similarity"] = [
    char_similarity(a, b)
    for a, b in zip(
        hard_feature_dataset["s1_address_norm"],
        hard_feature_dataset["s2_address_norm"]
    )
]

hard_feature_dataset["address_token_jaccard"] = [
    token_jaccard(a, b)
    for a, b in zip(
        hard_feature_dataset["s1_address_norm"],
        hard_feature_dataset["s2_address_norm"]
    )
]

hard_feature_dataset["country_exact"] = (
    hard_feature_dataset["s1_country"].fillna("").str.lower()
    ==
    hard_feature_dataset["s2_country"].fillna("").str.lower()
).astype(int)

hard_feature_dataset["translit_name_char_similarity"] = [
    char_similarity(a, b)
    for a, b in zip(
        hard_feature_dataset["s1_name_translit"],
        hard_feature_dataset["s2_name_translit"]
    )
]

print("Features calculated.")

Features calculated.


In [47]:
feature_columns = [
    "name_char_similarity",
    "name_3gram_jaccard",
    "name_token_jaccard",
    "address_char_similarity",
    "address_token_jaccard",
    "country_exact",
    "translit_name_char_similarity"
]

print(
    hard_feature_dataset[
        ["label"] + feature_columns
    ].groupby("label").mean()
)

       name_char_similarity  name_3gram_jaccard  name_token_jaccard  \
label                                                                 
0                  0.535683            0.204715            0.249696   
1                  0.795814            0.658732            0.623358   

       address_char_similarity  address_token_jaccard  country_exact  \
label                                                                  
0                     0.355360               0.024380            1.0   
1                     0.836225               0.776738            1.0   

       translit_name_char_similarity  
label                                 
0                           0.535907  
1                           0.828650  


hard-negative name similarity increased from our earlier random negatives

In [48]:
from sklearn.model_selection import GroupShuffleSplit

feature_columns = [
    "name_char_similarity",
    "name_3gram_jaccard",
    "name_token_jaccard",
    "address_char_similarity",
    "address_token_jaccard",
    "translit_name_char_similarity"
]

X = hard_feature_dataset[feature_columns].copy()
y = hard_feature_dataset["label"].copy()

# Replace missing similarity values with 0
X = X.fillna(0)

groups = hard_feature_dataset["s1_entity_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

print("\nTrain labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_val.value_counts())

Training: (63843, 6)
Validation: (16083, 6)

Train labels:
label
1    32101
0    31742
Name: count, dtype: int64

Validation labels:
label
0    8184
1    7899
Name: count, dtype: int64


logistic-regression baseline

In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logreg.fit(X_train, y_train)

val_prob = logreg.predict_proba(X_val)[:, 1]

print("ROC-AUC:", roc_auc_score(y_val, val_prob))
print("Average Precision:", average_precision_score(y_val, val_prob))

ROC-AUC: 0.9993175154136219
Average Precision: 0.999245168827289


In [50]:
import pandas as pd
import numpy as np

prob_summary = pd.DataFrame({
    "label": y_val.values,
    "probability": val_prob
})

print("Positive probability statistics:")
print(
    prob_summary.loc[
        prob_summary["label"] == 1,
        "probability"
    ].describe()
)

print("\nNegative probability statistics:")
print(
    prob_summary.loc[
        prob_summary["label"] == 0,
        "probability"
    ].describe()
)

Positive probability statistics:
count    7899.000000
mean        0.980303
std         0.094802
min         0.017566
25%         0.999840
50%         0.999998
75%         1.000000
max         1.000000
Name: probability, dtype: float64

Negative probability statistics:
count    8184.000000
mean        0.020127
std         0.076701
min         0.000149
25%         0.001600
50%         0.003538
75%         0.010706
max         0.999989
Name: probability, dtype: float64


decision layer

In [51]:
from sklearn.metrics import precision_score, recall_score, fbeta_score

thresholds = [
    0.50, 0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90, 0.92,
    0.94, 0.95, 0.96, 0.97, 0.98,
    0.985, 0.99, 0.995, 0.999
]

threshold_results = []

for threshold in thresholds:

    predictions = (val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        predictions,
        beta=0.5,
        zero_division=0
    )

    tp = ((predictions == 1) & (y_val.values == 1)).sum()
    fp = ((predictions == 1) & (y_val.values == 0)).sum()
    fn = ((predictions == 0) & (y_val.values == 1)).sum()

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "F0.5": f05,
        "TP": tp,
        "FP": fp,
        "FN": fn
    })

threshold_df = pd.DataFrame(threshold_results)

print(
    threshold_df.to_string(index=False)
)

 threshold  precision   recall     F0.5   TP  FP   FN
     0.500   0.992484 0.986327 0.991247 7791  59  108
     0.550   0.993610 0.984302 0.991734 7775  50  124
     0.600   0.994360 0.982150 0.991894 7758  44  141
     0.650   0.994859 0.979997 0.991851 7741  40  158
     0.700   0.995231 0.977592 0.991653 7722  37  177
     0.750   0.995601 0.974174 0.991240 7695  34  204
     0.800   0.996230 0.970123 0.990897 7663  29  236
     0.850   0.996597 0.964046 0.989912 7615  26  284
     0.900   0.996954 0.953159 0.987876 7529  23  370
     0.920   0.997471 0.948601 0.987298 7493  19  406
     0.940   0.997721 0.942398 0.986143 7444  17  455
     0.950   0.997710 0.937840 0.985133 7408  17  491
     0.960   0.997701 0.933916 0.984256 7377  17  522
     0.970   0.997825 0.929232 0.983308 7340  16  559
     0.980   0.998050 0.907203 0.978454 7166  14  733
     0.985   0.998180 0.902393 0.977429 7128  13  771
     0.990   0.998163 0.894164 0.975472 7063  13  836
     0.995   0.998126 0.8766